
## Databricks utilities

- File System utilities
- sceret utilities
- widget utilities
- notebook workflow utilities


# Databricks dbutils Revision — Gizmobox Project

## 1. dbutils.fs
- `ls(path)` — list files/dirs
- `mkdirs(path)` — create directory (with parents)
- `rm(path, recurse=True)` — delete file/dir
- `cp(source, dest, recurse=True)` — copy
- `mv(source, dest, recurse=True)` — move/rename
- `put(file, contents, overwrite=True)` — write small file directly
- `head(file, maxBytes)` — preview file content
- `mount(source, mountPoint, extraConfigs)` — mount external storage (deprecated in favor of Unity Catalog)
- `unmount(mountPoint)` — remove mount
- `mounts()` — list current mounts
- `refreshMounts()` — refresh mount cache on all nodes

## 2. dbutils.secrets
- `get(scope, key)` — retrieve secret value (shows [REDACTED] if printed)
- `list(scope)` — list keys in a scope
- `listScopes()` — list all scopes
- ⚠️ Secrets are CREATED via CLI/REST API (`secrets create-scope`, `secrets put`), NOT from the notebook

## 3. dbutils.widgets
- `text(name, defaultValue)` — free text input
- `dropdown(name, defaultValue, choices)` — fixed choice list
- `combobox(name, defaultValue, choices)` — dropdown + free text
- `multiselect(name, defaultValue, choices)` — pick multiple
- `get(name)` — retrieve current value
- `remove(name)` / `removeAll()` — clean up
- Used for parameterizing notebooks called from Jobs

## 4. dbutils.notebook
- `run(path, timeoutSeconds, arguments)` — run another notebook, returns its exit value; runs in a NEW ephemeral job/context (isolated, doesn't share variables)
- `exit(value)` — exit current notebook, optionally returning a string value to the caller
- Used for notebook orchestration/chaining without a full Job

## 5. dbutils.jobs (less common on exam, but exists)
- `taskValues.set(key, value)` — pass values between tasks in a multi-task Job
- `taskValues.get(taskKey, key, default, debugValue)` — retrieve value set by upstream task

## 6. Other utilities worth knowing
- `dbutils.library` — deprecated, install libraries (use `%pip` instead now)
- `dbutils.data.summarize(df)` — quick statistical summary of a DataFrame
- `dbutils.help()` — lists all dbutils modules and their functions (good for self-serve exploration)

### File System Utilities


In [0]:
display(dbutils.fs.ls('/'))

In [0]:
items = dbutils.fs.ls('dbfs:/databricks-datasets')
print(items)

In [0]:
folder_count = len([item for item in items if item.name.endswith('/')])
file_count = len([item for item in items if not item.name.endswith('/')])

print(f"Total Folder Count : {folder_count}")
print(f"Total File Count : {file_count}")

In [0]:
dbutils.fs.help()

In [0]:
display(dbutils.fs.ls('/Volumes/gizmobox/landing/operational_data/'))

In [0]:
dbutils.fs.mkdirs('abfss://gizmobox@dbstorageacc123.dfs.core.windows.net/landing/practice/custom_creation')

In [0]:
dbutils.fs.rm('abfss://gizmobox@dbstorageacc123.dfs.core.windows.net/landing/operational_data',recurse=True)

In [0]:
file = dbutils.fs.ls('/')
for details in file:
    print('-------START----------')
    print(details.path)
    print(details.name,"=====>>>name")
    print(details.size)


In [0]:
%sql
-- DROP VOLUME IF EXISTS gtesting.learning.pratice_data
CREATE VOLUME IF NOT EXISTS gtesting.learning.pratice_data


In [0]:
%sql

SHOW VOLUMES IN gtesting.learning


-- # # dbutils.fs.rm('/Volumes/gtesting/learning/practice_data/customers' ,recurse=True)

-- # # Use Databricks SQL to list all volumes in a catalog and schema
-- # # dg = spark.sql("SHOW VOLUMES IN gtesting.learning")
-- # # display(dg)
-- # %%sql SHOW VOLUMES IN gtesting.learning

In [0]:
dbutils.fs.ls('/Volumes/gtesting/learning/pratice_data')

In [0]:
dbutils.fs.cp('/Volumes/gizmobox/landing/operational_data/customers','/Volumes/gtesting/learning/pratice_data/customers',recurse=True)

In [0]:
display(dbutils.fs.ls('/Volumes/gtesting/learning/pratice_data/customers'));

In [0]:
dbutils.fs.mkdirs('/Volumes/gtesting/learning/pratice_data/special_customers')

dbutils.fs.mv('/Volumes/gtesting/learning/pratice_data/customers/customers_2025_01.json','/Volumes/gtesting/learning/pratice_data/special_customers')

In [0]:
display(dbutils.fs.head('/Volumes/gtesting/learning/pratice_data/special_customers/customers_2025_01.json'))

display(file_data)

In [0]:
dbutils.fs.head('/Volumes/gtesting/learning/pratice_data/special_customers/customers_2025_01.json', 500)

In [0]:
display(dbutils.fs.mounts());

In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.get(scope="my_scope",key="my_password")

In [0]:
dbutils.secrets.listScopes()

dbutils.secrets.list('my_scope')



####  Widgets(For Parameterization)


In [0]:
dbutils.widgets.text("customer_id", "1000", "Customer ID")

dbutils.widgets.text('name','konduru sai',"Name")

In [0]:
dbutils.widgets.remove('customer_id')

In [0]:
dbutils.widgets.dropdown("customer_id", "1000", ["1000", "1001", "1002"], "Customer ID")

In [0]:
print(dbutils.widgets.get("customer_id"))

In [0]:
dbutils.widgets.combobox("region", "us-east", ["us-east", "us-west", "eu-west"], "Region")

In [0]:
print(dbutils.widgets.get("region"))

In [0]:
dbutils.widgets.multiselect("categories", "electronics", ["electronics", "clothing", "books", "toys"], "Categories")

In [0]:
print(dbutils.widgets.get("categories").split(","))

In [0]:
dbutils.widgets.removeAll()

In [0]:
result = dbutils.notebook.run(
    "./3.1 Child Notebook",   # adjust path if it's in a different folder
    timeout_seconds=60,
    arguments={"customer_id": "9999", "date": "2025-07-20"}
)

print("Returned value from child notebook:", result)